# KagMCP — MCP no oficial para Kaggle

**KagMCP** es un servidor no oficial de Model Context Protocol para conectar ChatGPT u otros agentes compatibles a una sesión autorizada de Kaggle.

Integra **MCP sobre HTTPS con OAuth y PIN**, herramientas de archivos/terminal, acceso opcional a **GitHub con token personal** y acceso SSH/SFTP opcional mediante VPS. Se ejecuta mediante un binario Go y un auxiliar verificado de Cloudflare.

**Configuración recomendada: solo MCP.** La celda **Configurar KagMCP en Python** define los parámetros normales, deja `pin = ""` para que Go genere un PIN temporal seguro al arrancar `run`. Para usar MCP **no necesitas añadir ningún Kaggle Secret**, ni GPU, ni VPS. El notebook consulta Kaggle Secrets **solo si activas SSH** para obtener sus contraseñas.


## 1. Arquitectura

```text
ChatGPT / agente MCP -> HTTPS .gradio.live (respaldo .trycloudflare.com) -> Kaggle:127.0.0.1:8181/mcp
                                                    |
                                                 KagMCP Go
                                                    |-- archivos en /kaggle/working
                                                    |-- datasets en /kaggle/input (solo lectura)
                                                    |-- procesos/comandos del runtime
                                                    |-- GitHub REST con PAT opcional
                                                    \-- SSH/SFTP por VPS (desactivado por defecto)
```

El enlace HTTPS temporal usa `MCP_TUNNEL=auto`: **Gradio FRP oficial primero**, y tras `GRADIO_RETRIES` fallos comprobados, Cloudflare automáticamente. `MCP_TUNNEL=gradio` o `cloudflare` elige solo ese proveedor; `none` usa proxy propio. La URL se muestra en la celda de arranque. **No necesitas VPS ni GPU para MCP**.

La ruta `/kaggle/working/.kagmcp` guarda configuración **no sensible**, estado OAuth y caché; Kaggle no garantiza su persistencia entre sesiones.

La configuración Python establece explícitamente `SSH_ENABLED=false` por defecto, incluso si existían antiguas etiquetas SSH en Kaggle Secrets.


## 2. Clonar el repositorio remoto y compilar en Kaggle

No necesitas subir un ejecutable. **Ejecuta las siguientes tres celdas en orden** con Internet habilitado: clonar/actualizar `main` desde GitHub (sin sobrescribir cambios locales), comprobar/descargar Go y compilar el binario directamente en `/kaggle/working`. Si Go no está instalado o es anterior al mínimo de `go.mod`, se descarga de **go.dev** con comprobación del SHA-256 publicado por el equipo de Go. El binario se reconstruye cada vez que ejecutas la celda de compilación.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import re
import shutil
import subprocess
import tempfile
import urllib.request

REPO_URL = "https://github.com/ThowiLabs/kagssh-go.git"
WORKING = Path("/kaggle/working")
REPO_DIR = WORKING / "kagssh-go"
BINARY = WORKING / "kagmcp-linux-amd64"

if platform.system() != "Linux" or platform.machine().lower() not in ("x86_64", "amd64"):
    raise RuntimeError("Este ejemplo requiere Kaggle Linux x86_64 (amd64).")
if not WORKING.is_dir():
    raise RuntimeError("Ejecuta este notebook en Kaggle (/kaggle/working).")
if not shutil.which("git"):
    raise RuntimeError("No se encontró git en el entorno de Kaggle.")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)],
        check=True, timeout=180,
    )
else:
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"{REPO_DIR} existe pero no es un clon Git. Muévelo antes de continuar.")
    current_origin = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True
    ).strip().rstrip("/")
    if current_origin.removesuffix(".git").lower() != REPO_URL.removesuffix(".git").lower():
        raise RuntimeError(f"El directorio ya existe y su remoto es distinto: {current_origin}")
    dirty = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True
    ).strip()
    if dirty:
        raise RuntimeError("El clon tiene cambios sin guardar; no se actualizará ni se borrará nada.")
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "origin", "main"],
        check=True, timeout=180,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "merge", "--ff-only", "origin/main"],
        check=True, timeout=60,
    )

REVISION = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True
).strip()
print("Repositorio listo:", REPO_DIR)
print("Revisión remota clonada:", REVISION)

In [ ]:
# Usa el Go del sistema si cumple el mínimo de go.mod; si falta o es antiguo,
# instala la versión necesaria desde la distribución oficial con SHA-256 verificado.
mod_text = (REPO_DIR / "go.mod").read_text(encoding="utf-8")
match = re.search(r"(?m)^go\s+(\d+)\.(\d+)(?:\.(\d+))?\s*$", mod_text)
if not match:
    raise RuntimeError("No se encontró la versión mínima de Go en go.mod.")
MIN_GO = tuple(int(v or 0) for v in match.groups())
required = ".".join(str(v) for v in MIN_GO)
go_executable = shutil.which("go")

def installed_version(executable):
    if not executable:
        return None
    result = subprocess.run([executable, "version"], capture_output=True, text=True)
    found = re.search(r"go(\d+)\.(\d+)\.(\d+)", result.stdout)
    return tuple(map(int, found.groups())) if result.returncode == 0 and found else None

existing = installed_version(go_executable)
# Go 1.27.0 y 1.27.1 preceden a los parches del 8/oct/2026.
def secure_go(version):
    return (version is not None and version >= MIN_GO
            and (version[:2] != (1, 27) or version[2] >= 2))

if secure_go(existing):
    GO_BIN = Path(go_executable)
    print("Go instalado suficiente:", subprocess.check_output([str(GO_BIN), "version"], text=True).strip())
else:
    if not os.environ.get("KAGGLE_USER_SECRETS_TOKEN") and not os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        print("Nota: esta instalación necesita acceso a Internet desde Kaggle.")
    release_version = "go" + required
    catalog_url = "https://go.dev/dl/?mode=json&include=all"
    with urllib.request.urlopen(catalog_url, timeout=40) as response:
        releases = json.load(response)
    release = next((item for item in releases if item.get("version") == release_version), None)
    if release is None:
        raise RuntimeError(f"La versión {release_version} no figura en las descargas oficiales.")
    archive_meta = next(
        (f for f in release["files"] if f.get("os") == "linux"
         and f.get("arch") == "amd64" and f.get("kind") == "archive"), None
    )
    if archive_meta is None or not re.fullmatch(r"[0-9a-f]{64}", archive_meta.get("sha256", "")):
        raise RuntimeError("No se pudo obtener el SHA-256 oficial de Go linux-amd64.")
    tempdir = Path(tempfile.mkdtemp(prefix="kagssh-toolchain-", dir="/tmp"))
    archive = tempdir / archive_meta["filename"]
    address = "https://go.dev/dl/" + archive_meta["filename"]
    print("Descargando Go verificado:", archive_meta["filename"])
    digest = hashlib.sha256()
    with urllib.request.urlopen(address, timeout=120) as source, archive.open("wb") as target:
        while chunk := source.read(1024 * 1024):
            target.write(chunk)
            digest.update(chunk)
    if digest.hexdigest() != archive_meta["sha256"]:
        archive.unlink(missing_ok=True)
        raise RuntimeError("SHA-256 inválido: se rechaza la instalación.")
    subprocess.run(["tar", "-xzf", str(archive), "-C", str(tempdir)], check=True, timeout=120)
    archive.unlink(missing_ok=True)
    GO_BIN = tempdir / "go" / "bin" / "go"
    if not GO_BIN.is_file() or installed_version(str(GO_BIN)) != MIN_GO:
        raise RuntimeError("No se pudo validar la versión instalada de Go.")

print("Go seleccionado:", subprocess.check_output([str(GO_BIN), "version"], text=True).strip())
print("Versión mínima del repositorio:", required)

In [ ]:
# Compila el código clonado para el runtime Linux de Kaggle (sin CGO).
build_env = os.environ.copy()
build_env.update({"CGO_ENABLED": "0", "GOOS": "linux", "GOARCH": "amd64", "GOTOOLCHAIN": "local"})
subprocess.run(
    [str(GO_BIN), "build", "-buildvcs=false", "-trimpath", "-ldflags=-s -w",
     "-o", str(BINARY), "./cmd/kagssh"],
    cwd=str(REPO_DIR), env=build_env, check=True, timeout=600,
)
version = subprocess.run([str(BINARY), "-version"], capture_output=True, text=True, check=True)
print("Compilación terminada:", BINARY)
print("Versión binaria:", version.stdout.strip())
print("Go usado:", subprocess.check_output([str(GO_BIN), "version"], text=True).strip())

## 3. Configurar KagMCP con Python (ejecutar una vez por sesión)

En la siguiente celda, **edita solo los valores normales**: SSH activado o no, túnel `auto` (Gradio prioritario, respaldo Cloudflare), reintentos, puertos y host del VPS opcional. El PIN MCP se define con `pin = ""` (Go genera uno temporal de 20 caracteres al iniciar) o con un valor opcional de **6 a 128 caracteres**. Solo el PAT opcional se solicita con `getpass`. No compartas la salida de `run` que contiene el PIN generado.

- **Solo MCP:** deja `SSH_ENABLED=False` y `USE_GITHUB=False`; necesitas cero llamadas a Kaggle Secrets.
- **MCP con GitHub:** cambia `USE_GITHUB=True`; introduces el PAT de forma oculta en la sesión.
- **MCP con SSH:** cambia `SSH_ENABLED=True`, introduce `SSH_HOST` y autoriza **solo** los Kaggle Secrets `SSH_PASSWORD` (contraseña de VPS) y `SSH_LOGIN_PASSWORD` (contraseña del servidor SSH dentro de Kaggle).

La celda prepara `RUNTIME_ENV` para los dos procesos Go de validación y ejecución. Elimina **de esa copia del entorno** el token interno de Kaggle Secrets, evitando que Go consulte etiquetas adicionales. Los valores privados viven en memoria durante la sesión, no en Git ni en la configuración JSON local. Ejecuta esta celda de nuevo únicamente cuando cambies parámetros o comiences una sesión nueva.


In [ ]:
# CONFIGURACIÓN DE KAGMCP — EDITA SOLO LAS OPCIONES NO SECRETAS.
# Directorio del proyecto: /kaggle/working/kagssh-go
# Directorio operativo: /kaggle/working
import getpass
import os

# 1) Servicios: para usar únicamente MCP deja SSH_ENABLED = False.
MCP_ENABLED = True
SSH_ENABLED = False
MCP_TUNNEL = "auto"  # Gradio primero, Cloudflare después; también "gradio", "cloudflare", "none".
GRADIO_RETRIES = 3  # Intentos Gradio antes del respaldo (1 a 5).
MCP_PUBLIC_URL = ""        # Si MCP_TUNNEL == "none", define el dominio HTTPS.
MCP_LISTEN_PORT = 8181
USE_GITHUB = False         # True para pedir tu PAT privado en una entrada oculta.

# 2) SSH/VPS OPCIONAL: no se consulta Secrets si SSH_ENABLED es False.
SSH_HOST = ""              # IP o dominio público del VPS; no es una contraseña.
SSH_USER = "root"
SSH_PORT_REMOTE = 22
SSH_PORT_KAGGLE = 2223
SSH_PORT_LOCAL = 2224
SSH_LOGIN_USER = "root"
SSH_FINGERPRINT = ""       # Huella SHA256 del VPS verificada fuera de banda.

# 3) Preparar entorno del proceso; Go NO consultará Secrets automáticamente.
# No eliminamos variables globales de la sesión de Kaggle: solo la copia
# que se entrega a los procesos KagMCP.
RUNTIME_ENV = {
    k: v for k, v in os.environ.items()
    if not (k.startswith("MCP_") or k.startswith("SSH_") or k.startswith("GITHUB_"))
}
RUNTIME_ENV.pop("KAGGLE_USER_SECRETS_TOKEN", None)
RUNTIME_ENV.pop("KAGGLE_IAP_TOKEN", None)

RUNTIME_ENV.update({
    "MCP_ENABLED": str(MCP_ENABLED).lower(),
    "SSH_ENABLED": str(SSH_ENABLED).lower(),
    "MCP_TUNNEL": MCP_TUNNEL,
    "MCP_GRADIO_RETRIES": str(GRADIO_RETRIES),
    "MCP_LISTEN_PORT": str(MCP_LISTEN_PORT),
    "MCP_PUBLIC_URL": MCP_PUBLIC_URL,  # Vacío también anula una preferencia antigua.
})

# 4) El PIN lo establece Go al arrancar; vacío => PIN temporal seguro.
pin = ""
if pin and (not 6 <= len(pin) <= 128 or any(c in pin for c in "\r\n\x00")):
    raise ValueError("PIN inválido: utiliza de 6 a 128 caracteres sin saltos de línea.")
RUNTIME_ENV["MCP_ACCESS_PIN"] = pin
if not pin:
    print("PIN no definido: Go generará uno temporal y lo mostrará una sola vez en la celda run.")
del pin

# GitHub con token personal (sin GitHub App y sin consultar Kaggle Secrets).
if USE_GITHUB:
    github_pat = getpass.getpass("GitHub Personal Access Token (no visible): ")
    if not github_pat.strip():
        raise ValueError("USE_GITHUB=True requiere un PAT.")
    RUNTIME_ENV["GITHUB_TOKEN"] = github_pat
    del github_pat

if SSH_ENABLED:
    if not SSH_HOST.strip():
        raise ValueError("Para activar SSH primero define SSH_HOST (VPS).")
    RUNTIME_ENV.update({
        "SSH_HOST": SSH_HOST,
        "SSH_USER": SSH_USER,
        "SSH_PORT_REMOTE": str(SSH_PORT_REMOTE),
        "SSH_PORT_KAGGLE": str(SSH_PORT_KAGGLE),
        "SSH_PORT_LOCAL": str(SSH_PORT_LOCAL),
        "SSH_LOGIN_USER": SSH_LOGIN_USER,
    })
    if SSH_FINGERPRINT:
        RUNTIME_ENV["SSH_FINGERPRINT"] = SSH_FINGERPRINT
    # Kaggle Secrets solo se utiliza para estos dos passwords SSH.
    # No leer SSH_HOST_KEY, MCP_ACCESS_PIN, GITHUB_TOKEN ni otros Secrets.
    from kaggle_secrets import UserSecretsClient
    ssh_secrets = UserSecretsClient()
    for label in ("SSH_PASSWORD", "SSH_LOGIN_PASSWORD"):
        try:
            value = ssh_secrets.get_secret(label)
        except Exception:
            raise RuntimeError(
                f"No se pudo leer el Secret {label}. Autorízalo en Kaggle; "
                "si hay HTTP 429, evita reintentos repetidos."
            ) from None
        if not value:
            raise RuntimeError(f"El Secret SSH obligatorio {label} está vacío.")
        RUNTIME_ENV[label] = value
    del value, ssh_secrets

print("Entorno de KagMCP preparado en memoria.")
print("MCP activado:", MCP_ENABLED, "| SSH activado:", SSH_ENABLED, "| GitHub activado:", USE_GITHUB)
print("Directorio del repositorio:", REPO_DIR)
print("Directorio del runtime:", WORKING)
print("No se han mostrado ni escrito credenciales en archivos; permanecen en memoria.")


## 4. Kaggle Secrets — solo contraseñas SSH si habilitas el VPS

**No configures MCP_ENABLED, MCP_TUNNEL, SSH_ENABLED, MCP_ACCESS_PIN ni GITHUB_TOKEN como Kaggle Secrets para este notebook.** Python establece las opciones normales; Go genera un PIN temporal si `pin = ""` y Python solicita únicamente el PAT opcional mediante `getpass`.

| Modalidad | Kaggle Secrets consultados | Entrada privada en Python |
|---|---|---|
| Solo MCP (recomendado) | **Ninguno** | Go genera PIN temporal, o puedes definirlo en `pin` |
| MCP + GitHub | **Ninguno** | PIN opcional + GitHub PAT oculto |
| MCP + SSH opcional | `SSH_PASSWORD`, `SSH_LOGIN_PASSWORD` | PIN opcional (+ GitHub PAT opcional) |

Cuando `SSH_ENABLED=True`, crea y autoriza estas dos etiquetas en **Add-ons → Secrets**. Ningún otro Secret es leído por el notebook ni por el proceso Go que inicia.

Si defines el PIN debe tener **6–128 caracteres**. Si lo dejas vacío, Go genera uno aleatorio de 20 caracteres y lo muestra una única vez en `run`; también podrás cambiarlo desde la web (`/`) tras autenticarte, lo que revoca las autorizaciones OAuth anteriores. El GitHub PAT debe tener permisos mínimos sobre los repositorios deseados (`Contents: read` o `read and write`); no se crea ninguna GitHub App. Para SSH, verifica preferentemente `SSH_FINGERPRINT` de tu VPS por un canal independiente.

**Importante:** el servidor no mantiene un sandbox de comandos: clientes MCP autorizados pueden ejecutar acciones con los permisos del proceso Kaggle. `RUNTIME_ENV` y el proceso Go tienen credenciales en memoria; no las imprimas, compartas ni serialices.


## 5. Validar la configuración Python

Ejecuta primero **configurar** (`pin = ""` es válido y no solicita nada; Go generará el PIN durante `run`). La siguiente celda valida las opciones **sin volver a consultar Kaggle Secrets** y sin abrir túneles; recibe `RUNTIME_ENV` ya preparado.


In [ ]:
# Comprueba la configuración que Python preparó, sin abrir túneles ni leer Secrets.
if "RUNTIME_ENV" not in globals() or "MCP_ACCESS_PIN" not in RUNTIME_ENV:
    raise RuntimeError("Primero ejecuta la celda configurar.")
check = subprocess.run(
    [str(BINARY), "-check"],
    env=RUNTIME_ENV, cwd=str(WORKING),
    text=True,
    capture_output=True,
    timeout=180,
)
if check.returncode != 0:
    # Los errores de configuración no deben incluir contraseñas/PIN/PAT.
    raise RuntimeError(
        "Configuración inválida. Revisa las opciones de la celda configurar. "
        f"Detalle: {check.stderr.strip()[:600]}"
    )
print(check.stdout.strip())
print("Validado con el entorno Python; no se consultaron Kaggle Secrets adicionales.")


## 6. Iniciar KagMCP y mostrar logs en vivo

La siguiente celda utiliza **el mismo `RUNTIME_ENV`** que validaste. Go no volverá a leer Kaggle Secrets.

Cuando el túnel esté listo, verás la URL `https://...gradio.live/mcp` si funciona Gradio. Tras agotar los reintentos, el respaldo será `https://...trycloudflare.com/mcp`. Úsala en ChatGPT y autoriza con el PIN definido en `configurar` o el PIN temporal que Go muestra en los logs de `run`.

Para detenerlo, pulsa **Detener/Interruptar**: finaliza el proceso Go y sus túneles. Si modificas parámetros, vuelve a ejecutar **configurar → validación → run**.


In [ ]:
# Mantiene la celda de Kaggle en ejecución y muestra los logs de Go directamente.
# Usa Detener / Interruptar para cerrar KagMCP.
import subprocess

if "RUNTIME_ENV" not in globals() or "MCP_ACCESS_PIN" not in RUNTIME_ENV:
    raise RuntimeError("Primero ejecuta la celda configurar.")
print("Iniciando KagMCP. Pulsa Detener/Interruptar en Kaggle para desconectar.", flush=True)
with subprocess.Popen(
    [str(BINARY)],
    cwd=str(WORKING),
    env=RUNTIME_ENV,  # Entorno preparado en Python, sin token interno de Kaggle Secrets.
    stdin=subprocess.DEVNULL,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    errors="replace",
    # Se reenvía cada línea a print() de Jupyter, sin hilos ni segundo plano.
) as kagssh:
    try:
        for line in kagssh.stdout:  # Bucle bloqueante: mantiene activa la celda.
            print(line, end="", flush=True)  # Logs visibles en la misma celda.
        status = kagssh.wait()
    except KeyboardInterrupt:
        print("\nInterrupción de Kaggle: cerrando KagMCP...", flush=True)
        if kagssh.poll() is None:
            kagssh.terminate()
        try:
            kagssh.wait(timeout=10)
        except subprocess.TimeoutExpired:
            kagssh.kill()
            kagssh.wait()
        print("KagMCP detenido; túnel cerrado.", flush=True)
    else:
        if status != 0:
            raise RuntimeError(f"KagMCP terminó con error (código {status}). Revisa los logs anteriores.")
        print("KagMCP terminó normalmente.", flush=True)


## 7. Conectar ChatGPT o un cliente MCP

También verás `panel_web=https://...gradio.live/` (o `https://...trycloudflare.com/` cuando se use el respaldo). Abre esta URL raíz y usa el PIN de MCP para gestionar proyectos, memoria, tareas, historial, GitHub y la Skill Ponytail v2 siempre activa. Para cada proyecto, verifica primero repo Git y notebook; después crea su UI Gradio con versiones y dependencias bloqueadas. Vigila el disco para evitar readonly.

Cuando veas en los logs:

```text
KagMCP listo para conectar cliente MCP url=https://...gradio.live/mcp
```

Usa la URL HTTPS **completa** y autoriza OAuth con el PIN definido en `configurar` o el temporal generado por Go en los logs de `run`. Ese PIN **no** se lee desde Kaggle Secrets.

**SSH opcional:** solo si pusiste `SSH_ENABLED=True` y configuraste correctamente un VPS con `GatewayPorts clientspecified`, podrás utilizar `ssh -p 2223 usuario_kaggle@IP_VPS` o `sftp -P 2223 usuario_kaggle@IP_VPS`. Si no activas SSH, no se abre este acceso.


## 8. Herramientas disponibles

- `environment_info`: descubre el runtime Kaggle.
- `list_dir`, `read_file`, `write_file`: inspecciona y edita archivos. `/kaggle/input` es solo lectura; `/kaggle/working` admite escritura.
- `exec`: ejecuta comandos con límites de tiempo y salida, desde `/kaggle/working`.
- `github_status`, `github_repositories`, `github_branches`, `github_file_read`, `github_file_write`: consultas y commits GitHub con PAT opcional.

El acceso MCP es directo al runtime: después de autorizar, los comandos pueden ejecutarse con permisos elevados del proceso. Autoriza solo clientes de confianza.


## Límites y seguridad

- Proyecto independiente y no oficial; respeta las cuotas y duración de los runtimes Kaggle.
- **Solo MCP:** ningún Kaggle Secret consultado. SSH está explícitamente desactivado en Python. GPU no requerida.
- **Túnel:** `MCP_TUNNEL=auto` intenta Gradio FRP oficial (cliente Go nativo FRP 0.44 con dependencias Go fijadas, sin frpc ni Gradio Python) hasta `GRADIO_RETRIES=3` veces, comprueba una respuesta de salud real y pasa a Cloudflare si falla; si seleccionas `gradio` o `cloudflare`, se respeta la elección sin respaldo automático.
- **SSH opcional:** Python lee únicamente `SSH_PASSWORD` y `SSH_LOGIN_PASSWORD` desde Kaggle Secrets si lo activas.
- PIN MCP: `pin = ""` hace que **Go genere** un PIN temporal de 20 caracteres mostrado una sola vez en el log de `run`; alternativamente puedes definir un PIN de 6–128 caracteres. El PAT GitHub sí se introduce con `getpass`. No compartas el cuaderno con un PIN visible ni la salida que contiene el PIN generado.
- Python excluye `KAGGLE_USER_SECRETS_TOKEN` y `KAGGLE_IAP_TOKEN` del entorno **del proceso Go**, evitando búsquedas automáticas en Secrets durante `-check` y `run`. El notebook original mantiene su entorno intacto.
- No publiques una salida que contenga credenciales. Un proceso con permisos root y acceso directo al runtime no constituye aislamiento total de secretos.
- `/kaggle/working/.kagmcp` guarda el estado OAuth y opciones no secretas; no garantiza persistencia entre sesiones.
- El servidor necesita OAuth; Cloudflare termina TLS y la aplicación escucha en `127.0.0.1`.
- GitHub usa un PAT limitado, no una GitHub App. No se garantiza compatibilidad con todos los clientes MCP sin prueba de extremo a extremo.

Documentación: `README.md` y `USO_Y_CONFIGURACION.md` en [ThowiLabs/kagssh-go](https://github.com/ThowiLabs/kagssh-go).
